In [ ]:
# ============================================================
# Molecular Subtype Cox Model (Optimized)
# ============================================================
%matplotlib inline

import os, sys, time, pickle, warnings
warnings.filterwarnings("ignore")
from scipy.stats import ttest_ind
import numpy as np
import pandas as pd
import scipy.stats as st
from collections import Counter

import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (8, 6)

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw
from lifelines import KaplanMeierFitter

# -----------------------
# Output directories
# -----------------------
data_dir    = "./outputs/data"
models_dir  = "./outputs/models"
plots_dir   = "./outputs/plots"
metrics_dir = "./outputs/metrics"

for d in [data_dir, models_dir, plots_dir, metrics_dir]:
    os.makedirs(d, exist_ok=True)

# -----------------------
# Helpers
# -----------------------
def bins(n):
    return max(10, int(np.sqrt(n)))

def ci(x, alpha=0.05):
    """Mean and (1-alpha) t-interval; returns (mean, halfwidth, lo, hi)."""
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    n = len(x)
    m = float(np.mean(x))
    if n < 2:
        return round(m, 3), np.nan, np.nan, np.nan
    se = st.sem(x)
    tcrit = st.t.ppf(1 - alpha / 2, df=n - 1)
    hw = float(tcrit * se)
    return round(m, 3), round(hw, 3), round(m - hw, 3), round(m + hw, 3)

def timeit(start_time):
    t = time.time() - start_time
    return f"{t:.1f}s" if t < 60 else f"{t/60:.1f}min"

def harrell_cindex(y, pred):
    return concordance_index_censored(
        y["Event"], y["Progression Free Survival"], pred
    )[0]

def uno_cindex(y_train, y_test, pred):
    try:
        tau = np.percentile(y_train["Progression Free Survival"], 90)
        return concordance_index_ipcw(y_train, y_test, pred, tau=tau)[0]
    except ValueError:
        print("  ⚠️ Uno's C-index failed for this fold (censoring survival = 0), returning NaN")
        return np.nan

def harrell_cv_scorer(estimator, X, y):
    pred = estimator.predict(X)
    return harrell_cindex(y, pred)


# 1. Load Data and Model

In [ ]:

# ============================================================
# 1. Load Data
# ============================================================
X_molecular = pd.read_pickle("./X_clinical_molecular.pkl")
y_df        = pd.read_pickle("./y.pkl")

tmp = X_molecular.merge(
    y_df.drop(columns="Cohort"), left_index=True, right_index=True
)

X_all  = tmp.drop(columns=["Progression Free Survival", "Event"])
cohort = y_df["Cohort"]
X_all["Cohort"] = cohort

X_discovery = X_all[X_all["Cohort"] == "Discovery"].drop(columns="Cohort")
X_replicate = X_all[X_all["Cohort"] == "Replicate"].drop(columns="Cohort")

y_discovery = np.array(
    [(bool(e), float(t)) for e, t in tmp.loc[X_discovery.index, ["Event", "Progression Free Survival"]].values],
    dtype=[("Event", "?"), ("Progression Free Survival", "<f8")],
)
y_replicate = np.array(
    [(bool(e), float(t)) for e, t in tmp.loc[X_replicate.index, ["Event", "Progression Free Survival"]].values],
    dtype=[("Event", "?"), ("Progression Free Survival", "<f8")],
)

print("Discovery:", X_discovery.shape, "Replicate:", X_replicate.shape)
print("Discovery events:", int(np.sum(y_discovery["Event"])),
      "Replicate events:", int(np.sum(y_replicate["Event"])))

# 2. Discovery

## 2.1 Train and Test

In [ ]:

# ============================================================
# 2. Nested Cross-Validation
# ============================================================

# Uniform penalty across all molecular features
molecular_penalty = 1
penalty_factor = np.ones(X_discovery.shape[1], dtype=float) * molecular_penalty

# Adaptive folds based on event count
n_events = int(np.sum(y_discovery["Event"]))
if n_events < 10:
    n_splits = 3
elif n_events < 20:
    n_splits = 5
else:
    n_splits = 10

outer_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f"Using n_splits={n_splits} (events={n_events})")

l1_grid = np.arange(0.1, 1.01, 0.1).tolist()
n_alphas        = 100
alpha_min_ratio = 0.1
max_iter        = 50000
alpha_floor     = None
n_jobs          = 1

cv_rows = []
outer_s = time.time()

for fold_idx, (train_idx, test_idx) in enumerate(
    outer_cv.split(X_discovery, y_discovery["Event"]), start=1
):
    fold_s = time.time()

    X_train = X_discovery.iloc[train_idx, :]
    X_test  = X_discovery.iloc[test_idx, :]
    y_train = y_discovery[train_idx]
    y_test  = y_discovery[test_idx]

    inner_splits       = list(inner_cv.split(X_train, y_train["Event"]))
    best_model         = None
    best_inner_harrell = -np.inf
    best_params        = None

    for l1 in l1_grid:
        path_model = CoxnetSurvivalAnalysis(
            l1_ratio=l1,
            penalty_factor=penalty_factor,
            n_alphas=n_alphas,
            alpha_min_ratio=alpha_min_ratio,
            fit_baseline_model=True,
            max_iter=max_iter,
        )
        path_model.fit(X_train, y_train)
        alphas = np.array(path_model.alphas_, dtype=float)

        if alpha_floor is not None:
            alphas = alphas[alphas >= alpha_floor]
        if len(alphas) == 0:
            continue

        base = CoxnetSurvivalAnalysis(
            l1_ratio=l1,
            penalty_factor=penalty_factor,
            fit_baseline_model=True,
            max_iter=max_iter,
        )

        gcv = GridSearchCV(
            estimator=base,
            param_grid={"alphas": [[a] for a in alphas]},
            scoring=harrell_cv_scorer,
            cv=inner_splits,
            n_jobs=n_jobs,
            refit=True,
            error_score=np.nan,
            return_train_score=False,
        )
        gcv.fit(X_train, y_train)

        score = gcv.best_score_
        if score is None or (isinstance(score, float) and np.isnan(score)):
            continue

        score = float(score)
        if score > best_inner_harrell:
            best_inner_harrell = score
            best_model  = gcv.best_estimator_
            best_params = {
                "l1_ratio": float(l1),
                "alpha":    float(gcv.best_params_["alphas"][0]),
            }

    if best_model is None:
        raise RuntimeError(
            f"Fold {fold_idx}: all candidate fits failed. "
            f"Try increasing alpha_min_ratio or setting alpha_floor."
        )

    pred_test = best_model.predict(X_test)
    har = harrell_cindex(y_test, pred_test)
    uno = uno_cindex(y_train, y_test, pred_test)

    cv_rows.append({
        "fold":           fold_idx,
        "best_estimator": best_model,
        "inner_harrell":  best_inner_harrell,
        "harrell":        har,
        "uno":            uno,
        "best_l1_ratio":  best_params["l1_ratio"],
        "best_alpha":     best_params["alpha"],
    })

    print(
        f"Fold {fold_idx}/{n_splits} in {timeit(fold_s)} | "
        f"Harrell={har:.3f} Uno={uno} | "
        f"inner(Harrell)={best_inner_harrell:.3f} | "
        f"l1_ratio={best_params['l1_ratio']:.2f}, alpha={best_params['alpha']:.4g}"
    )

cv_results = pd.DataFrame(cv_rows)
print(f"\nNested CV finished in {timeit(outer_s)}")

n_jobs = -1

# Print and save CV metrics
scorings     = ["harrell", "uno"]
metrics_text = []
for scoring in scorings:
    c, u, l, h = ci(cv_results[scoring])
    metric_str  = f"{scoring}: {c}  CI: +/-{u}  [{l}, {h}]"
    print(metric_str)
    metrics_text.append(metric_str)

with open(os.path.join(metrics_dir, "cv_metrics.txt"), "w") as f:
    f.write("\n".join(metrics_text))

cv_results.to_csv(os.path.join(metrics_dir, "cv_results.csv"), index=False)

# 3. Final Model 


In [ ]:
# Most frequent l1_ratio across folds
l1_counts      = Counter(cv_results["best_l1_ratio"].tolist())
final_l1_ratio = l1_counts.most_common(1)[0][0]

# Median alpha from folds that used that dominant l1_ratio
final_alpha = float(
    cv_results.loc[cv_results["best_l1_ratio"] == final_l1_ratio, "best_alpha"].median()
)

print(f"\nl1_ratio counts across folds: {dict(l1_counts)}")
print(f"Final l1_ratio (dominant):              {final_l1_ratio}")
print(f"Final alpha (median of matching folds): {final_alpha:.4g}")

estimator = CoxnetSurvivalAnalysis(
    l1_ratio=final_l1_ratio,
    alphas=[final_alpha],
    penalty_factor=penalty_factor,
    fit_baseline_model=True,
    max_iter=max_iter,
)
estimator.fit(X_discovery, y_discovery)

final_coef = np.asarray(estimator.coef_).reshape(-1)
print(f"Nonzero features: {int(np.sum(np.abs(final_coef) > 1e-12))}")

with open(os.path.join(models_dir, "estimator.pkl"), "wb") as f:
    pickle.dump(estimator, f)
print(f"Saved estimator -> {models_dir}/estimator.pkl")

with open(os.path.join(metrics_dir, "final_hyperparameters.txt"), "w") as f:
    f.write(f"l1_ratio counts across folds: {dict(l1_counts)}\n")
    f.write(f"Final l1_ratio: {final_l1_ratio}\n")
    f.write(f"Final alpha: {final_alpha}\n")

# 4. Discovery Risk Scores & Stratification

In [ ]:
risk_discovery = pd.DataFrame(
    {"Risk Score": estimator.predict(X_discovery)},
    index=X_discovery.index
)
risk_discovery["Event"] = y_discovery["Event"].astype(int)
risk_discovery["Progression Free Survival"] = y_discovery["Progression Free Survival"].astype(float)

median_threshold = float(risk_discovery["Risk Score"].median())
risk_discovery["Risk Group"] = np.where(
    risk_discovery["Risk Score"] > median_threshold, "High", "Low"
)

with open(os.path.join(data_dir, "discovery_results.pkl"), "wb") as f:
    pickle.dump(risk_discovery, f)
with open(os.path.join(data_dir, "risk_threshold.pkl"), "wb") as f:
    pickle.dump({"median_threshold": median_threshold}, f)

#  Also save threshold as txt
with open(os.path.join(metrics_dir, "risk_thresholds.txt"), "w") as f:
    f.write(f"Risk categorization: Low vs High\n")
    f.write(f"Threshold (50th percentile/median): {median_threshold}\n")
    f.write(f"Low: Risk Score <= {median_threshold}\n")
    f.write(f"High: Risk Score > {median_threshold}\n")
    
# Also save as CSV
risk_discovery.to_csv(os.path.join(data_dir, "discovery_results.csv"), index=True)

print(f"Median threshold: {median_threshold}")

plt.figure(figsize=(8, 6))
risk_discovery["Risk Score"].hist(bins=bins(risk_discovery.shape[0]))
plt.xlabel("Risk Score")
plt.ylabel("Count")
plt.savefig(os.path.join(plots_dir, "discovery_risk_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

from scipy.stats import ttest_ind
high_scores = risk_discovery[risk_discovery["Risk Group"] == "High"]["Risk Score"]
low_scores  = risk_discovery[risk_discovery["Risk Group"] == "Low"]["Risk Score"]
t_stat, p_val = ttest_ind(high_scores, low_scores)
print(f"T-statistic: {t_stat:.4f}, p-value: {p_val:.4f}")

with open(os.path.join(metrics_dir, "ttest_results.txt"), "w") as f:
    f.write(f"Independent t-test (High vs Low Risk)\n")
    f.write(f"T-statistic: {t_stat}\n")
    f.write(f"p-value: {p_val}\n")
    f.write(f"Median threshold: {median_threshold}\n")

# 5. Bootstrap Uncertainty Quantification

In [ ]:

B    = 500
seed = 42
rng  = np.random.default_rng(seed)
tol  = 1e-12

feature_names = X_discovery.columns.to_list()
p             = len(feature_names)
coef_boot     = np.full((B, p), np.nan, dtype=float)
failed        = 0
n             = X_discovery.shape[0]

print(f"\nBootstrapping {B} fits...")

for b in range(B):
    idx = rng.integers(0, n, size=n)
    Xb  = X_discovery.iloc[idx, :]
    yb  = y_discovery[idx]

    model_b = CoxnetSurvivalAnalysis(
        l1_ratio=final_l1_ratio,
        alphas=[final_alpha],
        penalty_factor=penalty_factor,
        fit_baseline_model=False,
        max_iter=max_iter,
    )
    try:
        model_b.fit(Xb, yb)
        coef_boot[b, :] = np.asarray(model_b.coef_).reshape(-1)
    except Exception:
        failed += 1
        continue

print(f"Done. Failed fits: {failed}/{B} ({failed/B:.1%})")

ok      = ~np.isnan(coef_boot).all(axis=1)
coef_ok = coef_boot[ok, :]
B_ok    = coef_ok.shape[0]

if B_ok < 30:
    raise RuntimeError(f"Too few successful bootstrap fits ({B_ok}).")

sel_freq = (np.abs(coef_ok) > tol).mean(axis=0)
coef_med = np.nanmedian(coef_ok, axis=0)
coef_lo  = np.nanpercentile(coef_ok, 2.5, axis=0)
coef_hi  = np.nanpercentile(coef_ok, 97.5, axis=0)
hr_med   = np.exp(coef_med)
hr_lo    = np.exp(coef_lo)
hr_hi    = np.exp(coef_hi)

forest = pd.DataFrame({
    "Feature":             feature_names,
    "Selection_Frequency": sel_freq,
    "Coef_Median":         coef_med,
    "Coef_2.5%":           coef_lo,
    "Coef_97.5%":          coef_hi,
    "HR_Median":           hr_med,
    "HR_2.5%":             hr_lo,
    "HR_97.5%":            hr_hi,
})

final_nonzero              = np.abs(final_coef) > tol
forest["Nonzero_in_Final"] = final_nonzero
forest_filt                = forest[forest["Nonzero_in_Final"]].copy()
forest_filt                = forest_filt.sort_values("Selection_Frequency", ascending=False)
forest_filt.to_csv(os.path.join(data_dir, "bootstrap_forest.csv"), index=False)

if len(forest_filt) > 0:
    top  = forest_filt.head(25).copy().sort_values("HR_Median")
    ypos = np.arange(len(top))

    plt.figure(figsize=(9, 10))
    plt.errorbar(
        top["HR_Median"], ypos,
        xerr=[top["HR_Median"] - top["HR_2.5%"], top["HR_97.5%"] - top["HR_Median"]],
        fmt="o"
    )
    plt.axvline(1.0)
    plt.yticks(ypos, top["Feature"])
    plt.xlabel("Hazard Ratio (median, 95% bootstrap CI)")
    plt.title("Bootstrap forest plot — Molecular CoxNet")
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "bootstrap_forest_plot.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
else:
    print("⚠️ No nonzero features in final model — forest plot skipped.")

coef_vec = np.asarray(estimator.coef_).reshape(-1)
coef_df  = pd.DataFrame({"coef": coef_vec}, index=X_discovery.columns)
coef_df  = coef_df.loc[np.abs(coef_df["coef"]) > tol].copy()

if len(coef_df) > 0:
    coef_df["HR"] = np.exp(coef_df["coef"])
    coef_df = coef_df.sort_values("HR", ascending=False)
    coef_df.to_csv(os.path.join(data_dir, "coef_hr_table.csv"))
    print(coef_df)
else:
    print("⚠️ No nonzero coefficients in final model.")

In [ ]:
# ============================================================
# BOOTSTRAP + FOREST PLOT — MOLECULAR COXNET
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sksurv.linear_model import CoxnetSurvivalAnalysis
from lifelines import CoxPHFitter

# -----------------------
# Settings
# -----------------------
B    = 500
seed = 42
rng  = np.random.default_rng(seed)
tol  = 1e-12

feature_names = X_discovery.columns.to_list()
p             = len(feature_names)
n             = X_discovery.shape[0]
coef_boot     = np.full((B, p), np.nan, dtype=float)
failed        = 0

# -----------------------
# Bootstrap fits
# -----------------------
print(f"Bootstrapping {B} fits...")
for b in range(B):
    idx = rng.integers(0, n, size=n)
    Xb  = X_discovery.iloc[idx, :]
    yb  = y_discovery[idx]

    model_b = CoxnetSurvivalAnalysis(
        l1_ratio=final_l1_ratio,
        alphas=[final_alpha],
        penalty_factor=penalty_factor,
        fit_baseline_model=False,
        max_iter=max_iter,
    )
    try:
        model_b.fit(Xb, yb)
        coef_boot[b, :] = np.asarray(model_b.coef_).reshape(-1)
    except Exception:
        failed += 1
        continue

print(f"Done. Failed fits: {failed}/{B} ({failed/B:.1%})")

# -----------------------
# Bootstrap summaries
# -----------------------
ok      = ~np.isnan(coef_boot).all(axis=1)
coef_ok = coef_boot[ok, :]
B_ok    = coef_ok.shape[0]

if B_ok < 30:
    raise RuntimeError(f"Too few successful bootstrap fits ({B_ok}).")

print(f"Successful fits: {B_ok}/{B}")

sel_freq = (np.abs(coef_ok) > tol).mean(axis=0)
coef_med = np.nanmedian(coef_ok,           axis=0)
coef_lo  = np.nanpercentile(coef_ok,  2.5, axis=0)
coef_hi  = np.nanpercentile(coef_ok, 97.5, axis=0)
hr_med   = np.exp(coef_med)
hr_lo    = np.exp(coef_lo)
hr_hi    = np.exp(coef_hi)

# -----------------------
# Build forest table
# -----------------------
final_coef    = np.asarray(estimator.coef_).reshape(-1)
final_nonzero = np.abs(final_coef) > tol

forest = pd.DataFrame({
    "Feature"            : feature_names,
    "Selection_Frequency": sel_freq,
    "Coef_Median"        : coef_med,
    "Coef_2.5%"          : coef_lo,
    "Coef_97.5%"         : coef_hi,
    "HR_Median"          : hr_med,
    "HR_2.5%"            : hr_lo,
    "HR_97.5%"           : hr_hi,
    "Nonzero_in_Final"   : final_nonzero,
})

forest_filt = forest[forest["Nonzero_in_Final"]].copy()
forest_filt = forest_filt.sort_values("Selection_Frequency", ascending=False)

if len(forest_filt) == 0:
    print("No nonzero features in final model — stopping here.")
else:
    print(f"Selected features: {len(forest_filt)}")

    # -----------------------
    # Univariate Cox p-values
    # -----------------------
    uni_results  = []
    failed_feats = []

    for feat in forest_filt["Feature"].tolist():
        try:
            df_uni = pd.DataFrame({
                "T"    : y_discovery["Progression Free Survival"],
                "E"    : y_discovery["Event"].astype(int),
                "feat" : X_discovery[feat].values,
            })
            cph = CoxPHFitter()
            cph.fit(df_uni, duration_col="T", event_col="E")
            row = cph.summary.loc["feat"]
            uni_results.append({
                "Feature"   : feat,
                "HR_uni"    : round(float(row["exp(coef)"]),           3),
                "HR_lo_uni" : round(float(row["exp(coef) lower 95%"]), 3),
                "HR_hi_uni" : round(float(row["exp(coef) upper 95%"]), 3),
                "p_uni"     : float(row["p"]),
            })
        except Exception as e:
            failed_feats.append(feat)
            print(f"  FAILED univariate: {feat} -> {e}")

    print(f"Univariate fits — successful: {len(uni_results)}, failed: {len(failed_feats)}")

    uni_df      = pd.DataFrame(uni_results)
    forest_filt = forest_filt.merge(uni_df, on="Feature", how="left")

    def fmt_p(p):
        if pd.isna(p): return "NA"
        if p < 0.001:  return "p<0.001"
        else:          return f"p={p:.3f}"

    forest_filt["p_display"] = forest_filt["p_uni"].apply(fmt_p)
    forest_filt.to_csv(os.path.join(data_dir, "bootstrap_forest.csv"), index=False)
    display(
            forest_filt.sort_values("p_uni")[[
                "Feature", "Selection_Frequency",
                "HR_Median", "HR_2.5%", "HR_97.5%",
                "HR_uni", "p_display"
            ]]
        )

    # -----------------------
    # Forest plot
    # -----------------------
    top  = forest_filt.sort_values("Selection_Frequency", ascending=False).head(25).copy()
    top  = top.sort_values("HR_Median")
    ypos = np.arange(len(top))

    fig, ax = plt.subplots(figsize=(12, max(6, len(top) * 1.1)))

    for i, (_, row) in enumerate(top.iterrows()):
        ax.errorbar(
            row["HR_Median"], i,
            xerr=[[row["HR_Median"] - row["HR_2.5%"]],
                  [row["HR_97.5%"] - row["HR_Median"]]],
            fmt="o", color="steelblue", capsize=3, markersize=6
        )

    ax.axvline(1.0, color="gray", linestyle="--", linewidth=0.9)
    ax.set_yticks(ypos)
    ax.set_yticklabels(top["Feature"], fontsize=9)
    ax.set_xlabel("Hazard Ratio (median, 95% bootstrap CI)", fontsize=11)
    ax.set_title("Bootstrap Forest Plot — Molecular CoxNet", fontsize=13)

    # Widen x-axis for p-value annotations
    x_lo, x_hi = ax.get_xlim()
    ax.set_xlim(x_lo, x_hi * 1.25)
    x_right = ax.get_xlim()[1]

    for i, (_, row) in enumerate(top.iterrows()):
        ax.text(
            x_right * 0.98, i,
            row["p_display"],
            va="center", ha="right", fontsize=8
        )

    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "bootstrap_forest_plot.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # -----------------------
    # Coefficient / HR table
    # -----------------------
    coef_df = pd.DataFrame({"coef": final_coef}, index=X_discovery.columns)
    coef_df = coef_df.loc[np.abs(coef_df["coef"]) > tol].copy()
    coef_df["HR"] = np.exp(coef_df["coef"])
    coef_df = coef_df.sort_values("HR", ascending=False)
    coef_df.to_csv(os.path.join(data_dir, "coef_hr_table.csv"))
    print(coef_df)

In [ ]:
pip install forestplot

In [ ]:
# ============================================================
# FOREST PLOT USING forestplot PACKAGE
# ============================================================
import forestplot as fp

# -----------------------
# Prepare dataframe
# -----------------------
fp_df = forest_filt.copy()

fp_df["estimate"] = fp_df["HR_Median"]
fp_df["ll"]       = fp_df["HR_2.5%"]
fp_df["hl"]       = fp_df["HR_97.5%"]
fp_df["varlabel"] = fp_df["Feature"]
fp_df["p_uni"]    = fp_df["p_uni"].astype(float)

# CI string: "0.91 (0.74 to 1.05)"
fp_df["est_ci"] = (
    fp_df["HR_Median"].round(2).astype(str)
    + " ("
    + fp_df["HR_2.5%"].round(2).astype(str)
    + " to "
    + fp_df["HR_97.5%"].round(2).astype(str)
    + ")"
)

# Pre-formatted p-value string
fp_df["pval_display"] = fp_df["p_uni"].apply(
    lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
)

# Sort by HR
fp_df = fp_df.sort_values("estimate")

# -----------------------
# Plot
# -----------------------
ax = fp.forestplot(
    fp_df,
    estimate="estimate",
    ll="ll",
    hl="hl",
    varlabel="varlabel",
    pval="p_uni",
    starpval=False,
    ci_report=False,                      # we supply our own CI string
    annote=["est_ci"],                    # left table: our CI string
    annoteheaders=["HR (95% CI)"],
    rightannote=["pval_display"],         # right table: p-value
    right_annoteheaders=["P-value"],
    xlabel="Hazard Ratio (median, 95% bootstrap CI)",
    xline=1.0,
    table=True,                           # <-- enables the clean table layout
    figsize=(10, 5),

)

ax.set_title("Forest Plot — Penalized Coxnet", pad=10, fontsize=13)
plt.savefig(os.path.join(plots_dir, "bootstrap_forest_plot.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# 6. Replicate Evaluation

In [ ]:
pred_rep = estimator.predict(X_replicate)

har = harrell_cindex(y_replicate, pred_rep)
uno = uno_cindex(y_discovery, y_replicate, pred_rep)  

print(f"\nReplicate Harrell C-index: {har:.4f}")
print(f"Replicate Uno C-index:     {uno}")

with open(os.path.join(metrics_dir, "replicate_metrics.txt"), "w") as f:
    f.write(f"Harrell's C-index: {har:.6f}\n")
    f.write(f"Uno's C-index (IPCW weights from discovery): {uno}\n")


# 7. Replicate Risk Scores & Stratification

In [ ]:
with open(os.path.join(data_dir, "risk_threshold.pkl"), "rb") as f:
    median_threshold = pickle.load(f)["median_threshold"]

risk_replicate = pd.DataFrame(
    {"Risk Score": estimator.predict(X_replicate)},
    index=X_replicate.index
)
risk_replicate["Event"] = y_replicate["Event"].astype(int)
risk_replicate["Progression Free Survival"] = y_replicate["Progression Free Survival"].astype(float)

risk_replicate["Risk Group"] = np.where(
    risk_replicate["Risk Score"] > median_threshold, "High", "Low"
)

with open(os.path.join(data_dir, "replicate_results.pkl"), "wb") as f:
    pickle.dump(risk_replicate, f)

# ✅ Also save as CSV
risk_replicate.to_csv(os.path.join(data_dir, "replicate_results.csv"), index=True)

plt.figure(figsize=(8, 6))
risk_replicate["Risk Score"].hist(bins=bins(risk_replicate.shape[0]))
plt.xlabel("Risk Score")
plt.ylabel("Count")
plt.savefig(os.path.join(plots_dir, "replicate_risk_distribution.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
# ── T-test: Risk Group separation ────────────────────────────────────────
high_scores = risk_replicate[risk_replicate["Risk Group"] == "High"]["Risk Score"]
low_scores  = risk_replicate[risk_replicate["Risk Group"] == "Low"]["Risk Score"]

t_stat, p_val = ttest_ind(high_scores, low_scores)
print(f"T-statistic: {t_stat:.4f}, p-value: {p_val:.4f}")
print(f"High risk n={len(high_scores)}, Low risk n={len(low_scores)}")

with open(os.path.join(metrics_dir, "replicate_ttest_results.txt"), "w") as f:
    f.write(f"Independent t-test (High vs Low Risk) — Replicate\n")
    f.write(f"T-statistic: {t_stat}\n")
    f.write(f"p-value: {p_val}\n")
    f.write(f"Median threshold (from discovery): {median_threshold}\n")
    f.write(f"High risk n: {len(high_scores)}\n")
    f.write(f"Low risk n: {len(low_scores)}\n")

# 8. Combine & Save All Risk Scores


In [ ]:
risk = pd.concat([risk_discovery, risk_replicate], axis=0)
risk["Cohort"] = (
    ["Discovery"] * risk_discovery.shape[0] +
    ["Replicate"]  * risk_replicate.shape[0]
)
risk.to_csv(os.path.join(data_dir, "risk_scores.csv"), index=True)
print("Saved combined risk scores.")

In [ ]:
pip install plotly

In [ ]:
# ============================================================
# Secondary Analysis — Molecular CoxNet
#   Part A: link risk scores + clinical + subtype
#   Part B: NF1 germline vs somatic (adjusting for EOR + age)
#   Part C: risk score x molecular subtype (Sankey + summaries)
# FB / D3B — pLGG molecular survival
# ============================================================
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

import plotly.graph_objects as go

# -----------------------
# Config — EDIT THESE
# -----------------------
CLINICAL_XLSX = "/Users/batalf/Desktop/LGG_Paper_07_01_2026/Molecular_Subtype_Model/data/Molecular_Cohort_with_Clinical.xlsx"
RISK_CSV      = "./outputs/data/risk_scores.csv"   # produced by the main pipeline
OUT_DIR       = "./outputs/secondary"
DAYS_PER_MONTH = 30.417

# Join key: the index of risk_scores.csv must map to one of the clinical ID cols.
# The script auto-detects; override here if needed ("sample_id" or "SubjectID").
JOIN_KEY_OVERRIDE = None

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "tables"), exist_ok=True)


# ============================================================
# PART A — Load and link
# ============================================================
clin = pd.read_excel(CLINICAL_XLSX, "Molecular+Clinical")
risk = pd.read_csv(RISK_CSV, index_col=0)  # index = subject key, cols incl Risk Score/Event/PFS/Risk Group/Cohort

# --- clean clinical variables ---
# Age: days -> years
clin["age_years"] = clin["Age at Diagnosis (days)"] / 365.25

# Extent of resection -> binary GTR/NTR vs sub-total; drop non-informative
EOR_MAP = {
    "Gross/Near total resection": "GTR_NTR",
    "Partial resection":          "STR_biopsy",
    "Biopsy only":                "STR_biopsy",
}
clin["EOR"] = clin["Extent of Tumor Resection"].map(EOR_MAP)  # NA for Not Applicable/Unavailable

# --- NF1 germline / somatic flags from the raw subtype string ---
def has(s, token):
    return int(token.lower() in str(s).lower())

clin["NF1_germline"] = clin["molecular_subtype"].apply(lambda s: has(s, "NF1-germline"))
clin["NF1_somatic"]  = clin["molecular_subtype"].apply(lambda s: has(s, "NF1-somatic"))
clin["NF1_any"]      = ((clin["NF1_germline"] == 1) | (clin["NF1_somatic"] == 1)).astype(int)

def nf1_group(r):
    if r["NF1_germline"] and r["NF1_somatic"]: return "both"
    if r["NF1_germline"]:                      return "germline"
    if r["NF1_somatic"]:                       return "somatic"
    return "none"
clin["NF1_group"] = clin.apply(nf1_group, axis=1)

# --- link risk <-> clinical -------------------------------------------------
def pick_join_key(risk_idx, clin):
    if JOIN_KEY_OVERRIDE:
        return JOIN_KEY_OVERRIDE
    ridx = set(map(str, risk_idx))
    best, best_score = None, (-1, -1)
    for col in ["SubjectID", "sample_id"]:
        keys = clin[col].astype(str)
        ov = len(ridx & set(keys))
        uniq = keys.is_unique
        # rank on (overlap, then uniqueness) — a unique key avoids join explosions
        print(f"  join-key candidate {col!r}: {ov}/{len(ridx)} overlap, unique={uniq}")
        score = (ov, int(uniq))
        if score > best_score:
            best, best_score = col, score
    return best

key = pick_join_key(risk.index, clin)
print(f"Using join key: {key}")
risk.index = risk.index.astype(str)

# Collapse clinical to ONE row per join key (avoids many-to-many blow-up).
# Prefer specimen-matched rows, then drop rows with a missing/duplicate key.
clin_k = clin.copy()
clin_k["_key"] = clin_k[key].astype(str)
clin_k = clin_k[clin_k[key].notna()]
match_rank = {"specimen-matched": 0}
clin_k["_rank"] = clin_k["Clinical Match"].map(match_rank).fillna(1)
clin_k = (clin_k.sort_values("_rank")
                 .drop_duplicates("_key", keep="first")
                 .set_index("_key"))
assert clin_k.index.is_unique, "clinical key still not unique after dedupe"

df = risk.join(
    clin_k[["molecular_subtype", "age_years", "EOR",
            "Extent of Tumor Resection", "Chemotherapy", "Radiotherapy",
            "NF1_germline", "NF1_somatic", "NF1_any", "NF1_group", "short_histology"]],
    how="left",
)
n_unmatched = df["molecular_subtype"].isna().sum()
print(f"Linked {len(df)-n_unmatched}/{len(df)} risk rows to clinical "
      f"({n_unmatched} unmatched).")

# Standardize survival col names coming out of risk_scores.csv
df = df.rename(columns={"Progression Free Survival": "PFS_months"})
# Ensure numeric event
df["Event"] = df["Event"].astype(int)

df.to_csv(os.path.join(OUT_DIR, "tables", "linked_risk_clinical.csv"))


# ============================================================
# PART B — NF1 germline vs somatic prognosis (+ EOR, age)
# ============================================================
nf1 = df[df["NF1_any"] == 1].copy()
print("\n" + "="*60)
print("PART B — NF1-altered subset")
print("="*60)
print("NF1 group counts:")
print(nf1["NF1_group"].value_counts())
print("\nEvents by NF1 group:")
print(nf1.groupby("NF1_group")["Event"].agg(["size", "sum"]))

# --- Power warning (printed loudly) ---
n_evt = int(nf1["Event"].sum())
print(f"\n⚠️  NF1 subset n={len(nf1)}, events={n_evt}. "
      f"This is HYPOTHESIS-GENERATING ONLY — any multivariable Cox here "
      f"violates events-per-variable rules; treat p-values as descriptive.")

# --- KM: germline vs somatic vs both ---
kmf = KaplanMeierFitter()
plt.figure(figsize=(8, 6))
ax = plt.gca()
for grp, sub in nf1.groupby("NF1_group"):
    if len(sub) == 0:
        continue
    kmf.fit(sub["PFS_months"], sub["Event"], label=f"{grp} (n={len(sub)}, e={int(sub['Event'].sum())})")
    kmf.plot_survival_function(ax=ax, ci_show=False)
plt.xlabel("PFS (months)"); plt.ylabel("Progression-free probability")
plt.title("NF1-altered pLGG — PFS by germline/somatic status")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "plots", "nf1_km_by_group.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Log-rank: germline-only vs somatic-only (drop "both" for the clean contrast) ---
gs = nf1[nf1["NF1_group"].isin(["germline", "somatic"])].copy()
if gs["NF1_group"].nunique() == 2 and gs.groupby("NF1_group")["Event"].sum().min() > 0:
    g = gs[gs["NF1_group"] == "germline"]
    s = gs[gs["NF1_group"] == "somatic"]
    lr = logrank_test(g["PFS_months"], s["PFS_months"], g["Event"], s["Event"])
    print(f"\nLog-rank germline vs somatic: p={lr.p_value:.4f} "
          f"(test stat={lr.test_statistic:.3f})")
else:
    print("\nLog-rank skipped: a group has 0 events or <2 groups present.")

# --- Cox: somatic (ref=germline) adjusting for age + EOR, penalized for small n ---
# 'both' recoded as somatic+germline both =1 handled via two indicator design:
cox_df = nf1.copy()
cox_df["somatic_flag"]  = (cox_df["NF1_somatic"] == 1).astype(int)   # somatic present
cox_df["germline_flag"] = (cox_df["NF1_germline"] == 1).astype(int)  # germline present
cox_df["EOR_GTR"] = (cox_df["EOR"] == "GTR_NTR").astype(int)         # 1=GTR/NTR, 0=STR/biopsy
cox_df = cox_df.dropna(subset=["EOR", "age_years", "PFS_months", "Event"])

model_cols = ["somatic_flag", "germline_flag", "age_years", "EOR_GTR"]
# drop constant columns (can happen in tiny subsets)
model_cols = [c for c in model_cols if cox_df[c].nunique() > 1]
fit_df = cox_df[model_cols + ["PFS_months", "Event"]].copy()

print(f"\nCox model n={len(fit_df)}, events={int(fit_df['Event'].sum())}, "
      f"covariates={model_cols}")
try:
    cph = CoxPHFitter(penalizer=0.1, l1_ratio=0.0)   # ridge penalty stabilizes tiny-n fit
    cph.fit(fit_df, duration_col="PFS_months", event_col="Event")
    summ = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
    summ.columns = ["HR", "HR_lo95", "HR_hi95", "p"]
    print(summ.round(3).to_string())
    summ.round(4).to_csv(os.path.join(OUT_DIR, "tables", "nf1_cox_adjusted.csv"))
except Exception as e:
    print(f"Cox fit failed (expected possible with tiny n): {e}")


# ============================================================
# PART C — Risk score x molecular subtype
# ============================================================
print("\n" + "="*60)
print("PART C — Risk score x subtype")
print("="*60)

# Multi-hot alteration flags from subtype string (independent drivers)
ALTERATIONS = {
    "KIAA1549_BRAF": "KIAA1549-BRAF",
    "BRAF_V600E":    "BRAF V600E",
    "NF1":           "NF1",           # germline OR somatic
    "FGFR":          "FGFR",
    "RTK":           "RTK",
    "IDH":           "IDH",
    "MYB":           "MYB",
    "CDKN2A_B":      "CDKN2A/B",
    "other_MAPK":    "other MAPK",
}
for col, tok in ALTERATIONS.items():
    df[col] = df["molecular_subtype"].apply(lambda s: has(s, tok))

# Coarse single-label group for the Sankey (dominant driver, hierarchy)
def dominant_group(s):
    s = str(s).lower()
    if "kiaa1549" in s:   return "KIAA1549-BRAF"
    if "v600e" in s:      return "BRAF V600E"
    if "nf1" in s:        return "NF1"
    if "fgfr" in s:       return "FGFR"
    if "rtk" in s:        return "RTK"
    if "idh" in s:        return "IDH"
    if "myb" in s:        return "MYB"
    if "mapk" in s:       return "other MAPK"
    if "wildtype" in s:   return "wildtype"
    return "other"
df["subtype_group"] = df["molecular_subtype"].apply(dominant_group)

# --- Summary table: per-alteration risk & outcome profile ---
rows = []
for col in ALTERATIONS:
    sub = df[df[col] == 1]
    if len(sub) == 0:
        continue
    rows.append({
        "Alteration":      col,
        "n":               len(sub),
        "mean_risk":       round(sub["Risk Score"].mean(), 3),
        "pct_high_risk":   round((sub["Risk Group"] == "High").mean() * 100, 1),
        "events":          int(sub["Event"].sum()),
        "event_rate_%":    round(sub["Event"].mean() * 100, 1),
        "median_PFS_mo":   round(sub["PFS_months"].median(), 1),
    })
alt_summary = pd.DataFrame(rows).sort_values("mean_risk", ascending=False)
print("\nPer-alteration risk/outcome profile:")
print(alt_summary.to_string(index=False))
alt_summary.to_csv(os.path.join(OUT_DIR, "tables", "alteration_risk_profile.csv"), index=False)

# --- Boxplot: risk score by dominant subtype group ---
order = df.groupby("subtype_group")["Risk Score"].median().sort_values().index.tolist()
plt.figure(figsize=(10, 6))
data = [df[df["subtype_group"] == g]["Risk Score"].values for g in order]
plt.boxplot(data, labels=order, showfliers=False)
for i, g in enumerate(order, 1):
    v = df[df["subtype_group"] == g]["Risk Score"].values
    plt.scatter(np.random.normal(i, 0.06, len(v)), v, s=10, alpha=0.5)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Risk Score"); plt.title("Risk score distribution by molecular subtype")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "plots", "risk_by_subtype_box.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Static stacked bar: % High vs Low risk within each subtype (journal-friendly) ---
comp = (df.groupby("subtype_group")["Risk Group"]
          .value_counts(normalize=True).unstack().reindex(order))
comp = comp.reindex(columns=["Low", "High"]).fillna(0)
n_by = df["subtype_group"].value_counts().reindex(order)
plt.figure(figsize=(10, 6))
plt.bar(order, comp["Low"]*100,  label="Low risk",  color="#4C9F70")
plt.bar(order, comp["High"]*100, bottom=comp["Low"]*100, label="High risk", color="#C1524E")
for i, g in enumerate(order):
    plt.text(i, 102, f"n={int(n_by[g])}", ha="center", va="bottom", fontsize=8)
plt.axhline(50, color="gray", ls="--", lw=0.8)
plt.ylim(0, 108); plt.ylabel("% of subtype"); plt.xticks(rotation=45, ha="right")
plt.title("Risk-group composition by molecular subtype"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "plots", "risk_composition_by_subtype.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Sankey: subtype group -> risk group -> event status ---
def sankey(df, cohort=None, fname="sankey.html"):
    d = df if cohort is None else df[df["Cohort"] == cohort]
    d = d.dropna(subset=["subtype_group", "Risk Group"])
    subtypes = order
    risk_groups = ["Low", "High"]
    event_lbls  = ["No progression", "Progression"]
    labels = subtypes + [f"{r} risk" for r in risk_groups] + event_lbls
    idx = {l: i for i, l in enumerate(labels)}

    src, tgt, val = [], [], []
    # subtype -> risk group
    for st in subtypes:
        for rg in risk_groups:
            n = len(d[(d["subtype_group"] == st) & (d["Risk Group"] == rg)])
            if n:
                src.append(idx[st]); tgt.append(idx[f"{rg} risk"]); val.append(n)
    # risk group -> event
    for rg in risk_groups:
        for ev, lbl in [(0, "No progression"), (1, "Progression")]:
            n = len(d[(d["Risk Group"] == rg) & (d["Event"] == ev)])
            if n:
                src.append(idx[f"{rg} risk"]); tgt.append(idx[lbl]); val.append(n)

    fig = go.Figure(go.Sankey(
        node=dict(label=labels, pad=15, thickness=18),
        link=dict(source=src, target=tgt, value=val),
    ))
    ttl = "Subtype → Risk → Outcome" + (f" ({cohort})" if cohort else " (all)")
    fig.update_layout(title_text=ttl, font_size=12, width=900, height=600)
    fig.write_html(os.path.join(OUT_DIR, "plots", fname))
    try:
        fig.write_image(os.path.join(OUT_DIR, "plots", fname.replace(".html", ".png")), scale=2)
    except Exception as e:
        print(f"  (PNG export skipped: {e})")

sankey(df, None,        "sankey_all.html")
sankey(df, "Discovery", "sankey_discovery.html")
sankey(df, "Replicate", "sankey_replicate.html")
print("\nSankey diagrams written to", os.path.join(OUT_DIR, "plots"))

# --- Heatmap: alteration x risk-group counts ---
alt_cols = list(ALTERATIONS.keys())
heat = pd.DataFrame({
    "Low":  [((df[c] == 1) & (df["Risk Group"] == "Low")).sum()  for c in alt_cols],
    "High": [((df[c] == 1) & (df["Risk Group"] == "High")).sum() for c in alt_cols],
}, index=alt_cols)
heat["%High"] = (heat["High"] / (heat["Low"] + heat["High"]).replace(0, np.nan) * 100).round(1)
heat.to_csv(os.path.join(OUT_DIR, "tables", "alteration_by_riskgroup.csv"))
print("\nAlteration × risk-group:")
print(heat.to_string())

print("\n✅ Secondary analysis complete →", OUT_DIR)